Let's make a simple chatbot-harness using Langchain

In [1]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma4:12b-mlx",temperature=0,)
response = llm.invoke("who are you, kid?")
print("Gemma4_12b_local response: ", response.content,"\n")


Gemma4_12b_local response:  I don't have an age or a physical body, so I'm not a "kid" in the traditional sense. I am a large language model, trained by Google.

Think of me as a knowledgeable assistant you can chat with, ask questions, or brainstorm ideas with. How can I help you today? 



In [2]:
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage


In [3]:
class State(TypedDict):
    """chat history"""
    messages: Annotated[list[BaseMessage], add_messages]

class ChatAgent():
    
    SYSTEM_PROMPT = """You're a friendly assistant. Answer briefly and clearly."""
   
    def __init__(self, llm):
        self.llm = llm
        self.workflow = self._create_workflow()

    def _human_message(self, state) -> dict:
        user_input = input()
        message = HumanMessage(content=(user_input))
        print(f"{message.type}: {message.content}")
        return {"messages": [message]}

    async def _ai_message(self, state) -> dict:
        ai_response = await self.llm.ainvoke(state['messages'])
        print(f"{ai_response.type}: {ai_response.content}")
        return {"messages":[ai_response]}

    def _should_continue(self,state) -> str:
        EXIT_COMMANDS = {"quit", "exit", "bye"}
        if state["messages"][-1].content.lower() in EXIT_COMMANDS:
            return "end"
        return "continue"

    def _create_workflow(self) -> StateGraph:
        """building graph structure with LangGraph"""
        workflow = StateGraph(State)

        #add nodes
        workflow.add_node("human_message", self._human_message)
        workflow.add_node("ai_message", self._ai_message)

        #add sequence
        workflow.set_entry_point("human_message")
        workflow.add_conditional_edges("human_message", self._should_continue, {"continue": "ai_message", "end": END})
        workflow.add_edge("ai_message","human_message")
        
        return workflow.compile()
    
    async def run(self, system_prompt: str | None = None):
        initial_state = {"messages": [SystemMessage(content= system_prompt or self.SYSTEM_PROMPT)]}

        return await self.workflow.ainvoke(initial_state)
    

In [4]:
print("Hello! That's your chatbot!")
print("For exit please enter: quit, exit or bye")
print("-" * 50)

app = ChatAgent(llm)
await app.run(); #or we can use another system prompt here

Hello! That's your chatbot!
For exit please enter: quit, exit or bye
--------------------------------------------------
human: Hi, how are you?
ai: I'm doing well, thank you for asking! How can I help you today?
human: could you explain shortly Piphagor's theorem
ai: Pythagoras' theorem is a geometric formula used to find the missing length of a side in a **right-angled triangle**.

The formula is:
**$a^2 + b^2 = c^2$**

Here is what that means:
*   **$a$ and $b$** are the two shorter sides (the legs).
*   **$c$** is the longest side (the hypotenuse), which is opposite the right angle.

In simple terms, if you know the length of two sides, you can use this formula to calculate the third!
human: bye
